# Utilization Prediction

## a) Business Case

During the descriptives task, we defined three KPIs:

1) Utilization rate
2) Energy delivered per hour
3) Rate of registered user sessions

The business case behind these KPIs is defined as:

<br><br>
<center>
  <b style="font-size: 1.3em;">
    The prediction model enables the site operator to offer dynamic pricing to registered users to maximize revenue and energy throughput while smoothing demand.
  </b>
</center>
<br><br>

The model predicts the hourly utilization of a given site. This information can be used to offer dynamic pricing models: when utilization and energy delivered are low, prices can be lowered to encourage usage. Notifications can be sent to registered users, e.g. through an app, to increase engagement, improve site utilization and maximize energy throughput.


## b) Prediction Model

In [ ]:
import pickle
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import cm
import numpy as np
import seaborn as sns
import holidays
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.metrics import mean_squared_error, r2_score

# Load prepared dataframes from pickle files
data_path = "../data_processed/"

with open(data_path + "df_prepared.pkl", "rb") as f:
    df = pickle.load(f)

with open(data_path + "merged_df_prepared.pkl", "rb") as f:
    merged_df = pickle.load(f)

with open(data_path + "relevant_weather_data_prepared.pkl", "rb") as f:
    relevant_weather_data = pickle.load(f)

print("DataFrames loaded successfully:")
print(f"  df shape: {df.shape}")
print(f"  merged_df shape: {merged_df.shape}")
print(f"  relevant_weather_data shape: {relevant_weather_data.shape}")

We choose site 1 for this task

In [ ]:
print(merged_df.siteID.unique())
print(merged_df.stationID.dtype)
df_site_1 = merged_df[merged_df['siteID'] == '1']
df_site_1 = df_site_1.sort_values(by='connectionTime', ascending=True)
df_site_1.head()


In [ ]:
mask = df['charging_duration'] == df['session_duration']
anzahl = df[mask].shape[0]
print(f"Anzahl der Sitzungen, bei denen die Ladedauer gleich der Sitzungsdauer ist: {anzahl}")

In [ ]:
print(df_site_1['stationID'].nunique())

## Calculate Utilization for all hour where there was a session

In [ ]:
def make_site1_hourly_utilization(df,
                                  conn_col='connectionTime',
                                  disc_col='disconnectTime',
                                  station_col='stationID'):
    df = df.copy()
    df[conn_col] = pd.to_datetime(df[conn_col], errors='coerce')
    df[disc_col] = pd.to_datetime(df[disc_col], errors='coerce')

    n_stations = df[station_col].nunique()

    def row_hour_contrib(row):
        start = row[conn_col]
        end = row[disc_col]
        if pd.isna(start) or pd.isna(end) or end <= start:
            return []
        hours = pd.date_range(
            start=start.floor('H'),
            end=(end - pd.Timedelta(seconds=1)).floor('H'),
            freq='H'
        )
        out = []
        for h in hours:
            h_start = max(start, h)
            h_end = min(end, h + pd.Timedelta(hours=1))
            minutes = (h_end - h_start).total_seconds() / 60.0
            if minutes > 0:
                out.append((h, minutes))
        return out

    contrib = df.apply(row_hour_contrib, axis=1)
    tmp = pd.DataFrame(
        [{'datetime': h, 'minutes': m}
         for lst in contrib for (h, m) in lst]
    )

    if tmp.empty:
        hourly = pd.DataFrame(
            columns=['datetime', 'hourly_utilization_minutes',
                     'hourly_utilization']
        )
        hourly['datetime'] = pd.to_datetime(hourly['datetime'])
        return hourly.set_index('datetime')

    # 1) Aggregation: eine Zeile pro Stunde
    hourly = (
        tmp.groupby('datetime')['minutes']
        .sum()
        .rename('hourly_utilization_minutes')
        .to_frame()
    )

    hourly['hourly_utilization'] = (
        hourly['hourly_utilization_minutes'] / (n_stations * 60.0)
    ).clip(0, 1)



    return hourly



In [ ]:
# Compute hourly utilization for site 1 using connectionTime & disconnectTime
df_site_1_util = make_site1_hourly_utilization(df_site_1, 'connectionTime', 'disconnectTime')
df_site_1 = df_site_1.sort_values(by='connectionTime', ascending=True)


In [ ]:
df_site_1_util.head()


## Holidays

**pip install holidays runnen**

In [ ]:
US_holidays = holidays.US(state='CA')

df_site_1_util['is_holiday'] = df_site_1_util.index.to_series().apply(lambda dt: dt in US_holidays)
df_site_1_util['holiday_name'] = df_site_1_util.index.to_series().apply(lambda dt: US_holidays.get(dt) if dt in US_holidays else '')

df_site_1_util.head()



## Create hourly data of sessions

In [ ]:
def make_site1_hourly_features(df,
                               conn_col='connectionTime',
                               disc_col='disconnectTime',
                               station_col='stationID'):
    df = df.copy()
    df[conn_col] = pd.to_datetime(df[conn_col], errors='coerce')
    df[disc_col] = pd.to_datetime(df[disc_col], errors='coerce')

    n_stations = df[station_col].nunique()

    def row_hour_contrib(row):
        start = row[conn_col]
        end = row[disc_col]
        if pd.isna(start) or pd.isna(end) or end <= start:
            return []

        hours = pd.date_range(
            start=start.floor('H'),
            end=(end - pd.Timedelta(seconds=1)).floor('H'),
            freq='H'
        )

        out = []
        for h in hours:
            h_start = max(start, h)
            h_end = min(end, h + pd.Timedelta(hours=1))
            minutes = (h_end - h_start).total_seconds() / 60.0
            if minutes > 0:
                out.append({
                    'datetime': h,
                    'minutes': minutes,
                    'kWhDelivered': row['kWhDelivered'],
                    'userID': row['userID'],
                    'siteID': row[station_col],
                    'is_registered': row['isRegisteredUser'],
                    'session_duration': row['session_duration'],
                    'charging_duration': row['charging_duration']

                })
        return out

    contrib = df.apply(row_hour_contrib, axis=1)
    tmp = pd.DataFrame([item for lst in contrib for item in lst])

    if tmp.empty:
        return pd.DataFrame(columns=[
            'datetime', 'hourly_utilization_minutes', 'hourly_utilization'
        ]).set_index('datetime')

    # Stundenaggregation für Nutzung + zusätzliche Features
    hourly = (
        tmp.groupby('datetime')
            .agg(
                hourly_utilization_minutes=('minutes', 'sum'),
                energy_kwh_hour=('kWhDelivered', 'sum'),
                avg_kwh_per_session=('kWhDelivered', 'mean'),
                avg_session_duration=('session_duration', 'mean'),
                avg_charging_duration=('charging_duration', 'mean'),
                n_sessions=('userID', 'nunique'),
                n_registered_sessions=('is_registered', 'sum')
            )
    )

    hourly['hourly_utilization'] = (
        hourly['hourly_utilization_minutes'] / (n_stations * 60.0)
    ).clip(0, 1)

    return hourly

    

In [ ]:
df_site_1_features = make_site1_hourly_features(df_site_1, 'connectionTime', 'disconnectTime')
df_site_1_features.head()

We add a feature for holidays as this  influence the utilization due to the fact that employees do not work at this and university is also closed there. Additionally we add the already know time features from the session dataset.

In [ ]:
US_holidays = holidays.US(state='CA')

# Ensure index is datetime
df_site_1_features.index = pd.to_datetime(df_site_1_features.index)

df_site_1_features['is_holiday'] = df_site_1_features.index.to_series().apply(lambda dt: dt in US_holidays)
df_site_1_features['holiday_name'] = df_site_1_features.index.to_series().apply(lambda dt: US_holidays.get(dt) if dt in US_holidays else '')

#Add time features based on the index
df_site_1_features['year'] = df_site_1_features.index.year
df_site_1_features['month'] = df_site_1_features.index.month
df_site_1_features['day'] = df_site_1_features.index.day
df_site_1_features['hour'] = df_site_1_features.index.hour
df_site_1_features['dayofweek'] = df_site_1_features.index.dayofweek
df_site_1_features['is_weekend'] = df_site_1_features['dayofweek'].apply(lambda x: 1 if x >= 5 else 0)

def get_season(month):
    if month in [12, 1, 2]:
        return 'Winter'
    elif month in [3, 4, 5]:
        return 'Spring'
    elif month in [6, 7, 8]:
        return 'Summer'
    else:
        return 'Fall'
    
df_site_1_features['season'] = df_site_1_features['month'].apply(get_season)

In [ ]:
df_site_1_features.head()

In [ ]:
df_site_1_features.info()

In [ ]:
df_site_1_features['util_last_hour_flag'] = df_site_1_features['hourly_utilization'].shift(1)


df_site_1_features['util_prev_day_flag'] = df_site_1_features['hourly_utilization'].shift(24)

df_site_1_features.head()


In [ ]:
#lag_cols = ['util_last_hour_flag', 'util_prev_day_flag']
#df_site_1_features = df_site_1_features.dropna(subset=lag_cols)

In [ ]:
X = df_site_1_features[['hour', 'dayofweek', 'month']]#, 'util_last_hour_flag', 'util_prev_day_flag']]
y = df_site_1_features['hourly_utilization']

x_train, x_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=1)
x_train, x_hold, y_train, y_hold = train_test_split(x_train, y_train, test_size=(0.2/0.7), random_state=1)

In [ ]:
degrees = range(1, 6)
alphas = [0.01, 0.1, 1, 10]
models = ["linear", "Ridge", "Lasso"]
results = []

In [ ]:
for model in models:
    for degree in degrees:
        # create polynomial features
        poly = PolynomialFeatures(degree=degree, include_bias=False)
        X_train_poly = poly.fit_transform(x_train)
        X_hold_poly = poly.transform(x_hold)

        # standardize
        scaler = StandardScaler()
        X_train_scaled = scaler.fit_transform(X_train_poly)
        X_hold_scaled = scaler.transform(X_hold_poly)

        # for Ridge or Lasso, loop over alphas
        if model in ["Ridge", "Lasso"]:
            for alpha in alphas:
                # initialize
                if model == "Ridge":
                    reg = Ridge(alpha=alpha)
                else:
                    reg = Lasso(alpha=alpha, max_iter=10000)

                # fit
                reg.fit(X_train_scaled, y_train)

                # predict on validation and calculate error
                y_hold_pred = reg.predict(X_hold_scaled)
                mse_val = mean_squared_error(y_hold, y_hold_pred)
                r2_val = r2_score(y_hold, y_hold_pred)

                results.append({
                    "model": model,
                    "degree": degree,
                    "alpha": alpha,
                    "mse": mse_val,
                    "r2": r2_val,
                    "poly": poly,
                    "scaler": scaler,
                    "reg": reg
                })

        # linear regression without regularization
        else:
            # initialize
            reg = LinearRegression()

            # fit
            reg.fit(X_train_scaled, y_train)

            # predict on validation and calculate error
            y_hold_pred = reg.predict(X_hold_scaled)
            mse_val = mean_squared_error(y_hold, y_hold_pred)

            results.append({
                "model": model,
                "degree": degree,
                "alpha": None,
                "mse": mse_val,
                "poly": poly,
                "scaler": scaler,
                "reg": reg
            })

results = pd.DataFrame(results)

results

In [ ]:
best_idx = results["r2"].idxmax()
best_model = results.loc[best_idx]

print(best_model)

## c) Concrete Example of Application

TODO